# O₂ Dissociation Barrier on Pt(111) and at the Pt(211) Step

> **Ž. Šljivančanin, B. Hammer**
> "Oxygen dissociation at close-packed Pt terraces, Pt steps, and Ag-covered Pt steps studied with density functional theory"
> Surface Science 515, 235 (2002)
> [DOI: 10.1016/S0039-6028(02)01908-8](https://doi.org/10.1016/S0039-6028(02)01908-8)

Computes the O₂ dissociation barrier on flat Pt(111) and at the Pt(211) step from the four materials created in the
[structure notebook](defect_surface_step_platinum.ipynb): on each surface the O₂ precursor and the dissociated O pair are
relaxed if `RELAX` is set, and a climbing-image nudged elastic band (NEB) runs between them. The barrier, measured from
the precursor, and the O–O distance at the highest image are compared with Šljivančanin & Hammer's Table 1: 0.6 eV and
2.4 Å on Pt(111) (top–fcc–bridge), 0.9 eV and 3.0 Å at the Pt(211) step (top–bridge–top).

- The paper finds each transition state by relaxations at fixed O–O distances; here it is a climbing-image NEB between the precursor and the dissociated pair.
- If `RELAX` is set, the end points are MACE minima; the energies and the barrier come from the RPBE NEB. All atoms are free in both: the platform's NEB keeps no fixed atoms (the paper fixes the bottom half of each slab).
- Pt is at Standata's lattice constant, 3.943 Å (paper: 4.02 Å).
- The slabs are thinner than the paper's (paper: 4 Pt(111) layers of 9 Pt, 12 Pt(211) planes of 3 Pt); the structure notebook sets their size.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_surface_step_platinum.ipynb), which saves `Pt(111) O2 precursor`, `Pt(111) 2O dissociated`, `Pt(211) step O2 precursor` and `Pt(211) step 2O dissociated` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults); `RELAX = True` relaxes each material with MACE first and saves it as `<name> relaxed <MACE_MODEL_FAMILY> <MACE_MODEL>`.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the four materials by name, print their atom counts, then save them to the platform.
1. Configure the shared DFT model: one model for every workflow below.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Relax the four materials with MACE if `RELAX`, and save them to the platform.
1. Run one climbing-image NEB per surface between the precursor and the dissociated pair.
1. Retrieve the results: the energy profiles, the barriers and the O–O distance at the highest image.
1. Compare with Šljivančanin & Hammer (2002).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.calculators.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Materials and the paper's values

In [ ]:
# Names saved by defect_surface_step_platinum.ipynb, the k-grid of each surface, and Šljivančanin & Hammer 2002,
# Table 1: the barrier from the O2 precursor (eV) and the O–O distance at the transition state (Å).
# paper: 6x4x1 k-points on its own, smaller cells; [2, 2, 1] runs fast, raise it for a converged barrier.
SURFACES = {
    "Pt(111)": {"precursor": "Pt(111) O2 precursor", "dissociated": "Pt(111) 2O dissociated",
                "kgrid": [2, 2, 1], "paper_barrier": 0.6, "paper_transition_state_oxygen_distance": 2.4},
    "Pt(211)": {"precursor": "Pt(211) step O2 precursor", "dissociated": "Pt(211) step 2O dissociated",
                "kgrid": [2, 2, 1], "paper_barrier": 0.9, "paper_transition_state_oxygen_distance": 3.0},
}

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

NEB_WORKFLOW_SEARCH_TERM = "neb.json"
APPLICATION_NAME = "espresso"
# RPBE, set in 1.4, needs Quantum ESPRESSO 7.x
APPLICATION_VERSION = "7.5"

# True: the four end points relaxed with MACE before the NEB; False: the NEB on the structures as built
RELAX = True
MACE_MODEL_FAMILY = "MACE-MP-0"
MACE_MODEL = "large"
# the paper's RPBE has no dispersion correction
MACE_DISPERSION = False
MACE_DEFAULT_DTYPE = "float64"
MACE_DEVICE = "cpu"
# all atoms free, until the largest force on an atom is below the paper's 0.1 eV/Å
FMAX = 0.1  # eV/Å

CLUSTER_NAME = None
# OR allows one 16-core node, measured 79 min per ionic step on 50 atoms; OF runs one job on up to 10 nodes
QUEUE_NAME = QueueName.OF
NODES = 4
PPN = 40
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
# paper: RPBE with Vanderbilt ultrasoft pseudopotentials at 25 Ry; ours GBRV ultrasoft at its own converged 40/200 Ry
# selects the GBRV PBE pseudopotentials; QE_SYSTEM_SETTINGS sets the functional to RPBE
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"
ECUTWFC = 40   # Ry
ECUTRHO = 200  # Ry
# &SYSTEM of the NEB: RPBE as the paper; the adsorbed states without spin, as the paper
# (the NEB workflow defaults to nspin = 2); smearing is not stated in the paper
QE_SYSTEM_SETTINGS = {"input_dft": "rpbe", "nspin": 1, "occupations": "smearing", "smearing": "mv", "degauss": 0.01}
# &ELECTRONS of the NEB: the metal-slab mixing of the graphene/Ni(111) notebook; plain mixing takes 30–63 SCF
# iterations per step on these slabs
QE_ELECTRONS_SETTINGS = {"mixing_mode": "local-TF", "mixing_beta": 0.2}
# images between the end points; the path is converged when the largest image error is below 0.1 eV/Å
N_IMAGES = 1
# neb.x stops here, converged or not; the largest image error is printed in 9
NEB_MAXIMUM_ITERATIONS = 50
NEB_PATH_SETTINGS = {"nstep_path": NEB_MAXIMUM_ITERATIONS, "minimum_image": True}
MODEL_TAG = f"{APPLICATION_NAME}-{APPLICATION_VERSION} {FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry " + (
    " ".join(f"{key}={value}" for key, value in QE_SYSTEM_SETTINGS.items()))

NEB_UNIT = "neb"
# Name the workflow: a job whose workflow has the same name and k-grid on the same material is reused.
NEB_TAG = f"{MODEL_TAG} nImages={N_IMAGES} " + " ".join(f"{key}={value}" for key, value in NEB_PATH_SETTINGS.items())
# written by neb.x: every image of the final path
NEB_PATH_FILE = "__prefix__.xyz"
# written by neb.x: energy and error of every image of the final path
NEB_ENERGY_FILE = "__prefix__.dat"

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load the four materials from the uploads folder or the platform, and print provenance

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import load_material

material_names = [parameters[role] for parameters in SURFACES.values() for role in ("precursor", "dissociated")]
materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in material_names}
for name, material in materials_by_name.items():
    material.basis.set_labels_from_list([])
    print(f"{name}: {material.basis.number_of_atoms} atoms")

### 3.2. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {name: get_or_create_material(client, material, ACCOUNT_ID)
                   for name, material in materials_by_name.items()}

## 4. Configure the shared DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, nodes=NODES, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, nodes: {NODES}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Relaxation of the four materials with MACE

Runs only if `RELAX`: each material is relaxed with the MACE force field of 1.3 and saved to the platform; the relaxed materials are the end points of the NEB.

In [ ]:
from mat3ra.notebooks_utils.calculators.mlff import create_mlff_calculator
from mat3ra.notebooks_utils.workflows.relaxation import relax_material

if RELAX:
    calculator = create_mlff_calculator("mace", {
        "family": MACE_MODEL_FAMILY, "model": MACE_MODEL, "dispersion": MACE_DISPERSION,
        "default_dtype": MACE_DEFAULT_DTYPE, "device": MACE_DEVICE,
    })
    for name, material in materials_by_name.items():
        relaxed_material = relax_material(material, calculator, fmax=FMAX)
        relaxed_material.name = f"{name} relaxed {MACE_MODEL_FAMILY} {MACE_MODEL}"
        saved_materials[name] = get_or_create_material(client, relaxed_material, ACCOUNT_ID)

## 7. Nudged elastic band per surface

The precursor is the first image and the dissociated pair the last, both relaxed if `RELAX`; neb.x interpolates `N_IMAGES` images between them and climbs the highest one to the saddle point. Both end points list their atoms in the same order, so each atom moves to its own position.

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_materials_set

neb_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    NEB_WORKFLOW_SEARCH_TERM
)
end_points = {name: Material.create(saved) for name, saved in saved_materials.items()}
neb_job_ids = {}
new_neb_job_ids = []
for surface, parameters in SURFACES.items():
    path_materials = [end_points[parameters["precursor"]], end_points[parameters["dissociated"]]]
    materials_set_name = f"{surface} O2 dissociation {path_materials[0].id}-{path_materials[1].id}"
    materials_set = get_or_create_materials_set(client, ACCOUNT_ID, materials_set_name, path_materials, is_ordered=True)
    workflow = Workflow.create(neb_workflow_config)
    workflow.name = f"Nudged Elastic Band {materials_set_name} {NEB_TAG}"
    for subworkflow in workflow.subworkflows:
        for application in (subworkflow.application, *(unit.application for unit in subworkflow.units)):
            application.version = APPLICATION_VERSION
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=NEB_UNIT)
    apply_scf_kgrid(workflow, parameters["kgrid"], material=path_materials[0], unit_name=NEB_UNIT)
    patch_workflow_qe_input(workflow, {"system": QE_SYSTEM_SETTINGS, "electrons": QE_ELECTRONS_SETTINGS,
                                       "path": NEB_PATH_SETTINGS}, [NEB_UNIT])
    neb_unit = workflow.subworkflows[0].get_unit_by_name(name=NEB_UNIT)
    neb_unit.add_context({"name": "neb", "isEdited": True, "data": {"nImages": N_IMAGES}, "extraData": {}})
    job = find_job_for_material(client, path_materials[0].id, workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=parameters["kgrid"], unit_name=NEB_UNIT)
    if job is None:
        job = create_job(
            api_client=client, materials=path_materials, workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"Nudged Elastic Band {surface} {timestamp}",
            materials_set=materials_set,
        )
        new_neb_job_ids.append(job["_id"])
        print(f"✅ {surface}: created NEB job {job['_id']}")
    else:
        print(f"♻️  {surface}: reusing existing NEB job {job['_id']}")
    neb_job_ids[surface] = job["_id"]

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if new_neb_job_ids:
    submit_jobs(client.jobs, new_neb_job_ids)
    print(f"✅ Submitted {len(new_neb_job_ids)} NEB job(s).")
await wait_for_jobs_to_finish_async(client.jobs, list(neb_job_ids.values()), poll_interval=POLL_INTERVAL)

## 8. Retrieve the results
### 8.1. Energy profiles and barriers

The profile gives the energy of each image relative to the first one, the precursor; the barrier is its highest value.

In [ ]:
energy_profiles = {surface: client.properties.get_for_job(job_id, property_name="reaction_energy_profile")[0]
                   for surface, job_id in neb_job_ids.items()}
barriers = {surface: max(profile["yDataSeries"][0]) for surface, profile in energy_profiles.items()}

### 8.2. O–O distance at the climbing image and the largest image error

In [ ]:
from io import StringIO
from ase.io import read
from mat3ra.notebooks_utils.io import read_from_url

oxygen_distances, largest_image_errors = {}, {}
for surface, job_id in neb_job_ids.items():
    files = client.jobs.list_files(job_id)
    path_file = next(file for file in files if file["key"].endswith(NEB_PATH_FILE))
    energy_file = next(file for file in files if file["key"].endswith(NEB_ENERGY_FILE))
    images = read(StringIO(await read_from_url(path_file["signedUrl"])), index=":", format="xyz")
    energy_table = await read_from_url(energy_file["signedUrl"])
    largest_image_errors[surface] = max(float(line.split()[2]) for line in energy_table.splitlines()[1:-1])
    energies = energy_profiles[surface]["yDataSeries"][0]
    highest_image = images[energies.index(max(energies))]
    highest_image.set_cell(end_points[SURFACES[surface]["precursor"]].lattice.vector_arrays)
    highest_image.pbc = True
    oxygen_indices = [atom.index for atom in highest_image if atom.symbol == "O"]
    oxygen_distances[surface] = highest_image.get_distance(*oxygen_indices, mic=True)

## 9. Comparison with Šljivančanin & Hammer (2002)

In [ ]:
for surface, parameters in SURFACES.items():
    paper_barrier = parameters["paper_barrier"]
    paper_transition_state_oxygen_distance = parameters["paper_transition_state_oxygen_distance"]
    deviation_barrier = 100 * (barriers[surface] / paper_barrier - 1)
    deviation_distance = 100 * (oxygen_distances[surface] / paper_transition_state_oxygen_distance - 1)
    print(f"{surface} barrier: {barriers[surface]:.3f} eV, Šljivančanin & Hammer {paper_barrier} eV "
          f"({deviation_barrier:+.1f} %), largest image error {largest_image_errors[surface]:.3f} eV/Å")
    print(f"{surface} O–O at the climbing image: {oxygen_distances[surface]:.2f} Å, "
          f"Šljivančanin & Hammer {paper_transition_state_oxygen_distance} Å ({deviation_distance:+.1f} %)")

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

figure, axes = plt.subplots()
for surface, profile in energy_profiles.items():
    (line,) = axes.plot(profile["xDataArray"], profile["yDataSeries"][0], marker="o", label=surface)
    axes.axhline(SURFACES[surface]["paper_barrier"], linestyle="--", color=line.get_color())
axes.set_xlabel("Reaction coordinate")
axes.set_ylabel("E − E(O₂ precursor) (eV)")
axes.set_title("NEB energy profiles; dashed: Šljivančanin & Hammer (2002) barriers")
axes.legend()
display_matplotlib_figure(figure)

## References

[1] Šljivančanin, Ž., & Hammer, B. (2002). Oxygen dissociation at close-packed Pt terraces, Pt steps, and Ag-covered Pt steps studied with density functional theory. Surface Science, 515(1), 235–244. https://doi.org/10.1016/S0039-6028(02)01908-8